# Aurelium - Phase 2: Citation Impact Prediction

This notebook implements **Phase 2: Citation Prediction** for Project Aurelium.

### Goal
Predict the expected 3-year citation count for newly published AI papers using strictly publication-time features:
- **Text Representations**: 768-dimensional L2-normalized SPECTER embeddings of `"title [SEP] abstract"`
- **Metadata**: Number of references, text lengths, publication month, paper type, and taxonomy categoricals (field, subfield, topic)

### Design Rules & Leakage Constraints
1. **Target**: $y = \sum_{t \in [y_{\text{pub}}, y_{\text{pub}}+2]} \text{cited\_by\_count}(t)$. Trained on $\log(1 + y)$ to compress heavy-tailed power-law skew.
2. **Strict Time-Based Split**: Train (2015–2020), Validation (2021), Test (2022–2023). No random splits (prevents future information leakage).
3. **Zero Target Leakage**: `counts_by_year` is isolated strictly to label generation. Cumulative counters (`cited_by_count`, `fwci`) are completely excluded.
4. **Strict Fit Policy**: Scalers, One-Hot encoders, and PCA are fit **strictly on the training split**.

In [ ]:
# 1. Environment & Dataset Path Resolution
import os
import sys
from pathlib import Path
import torch

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Locate papers.jsonl from Kaggle input or local repository
kaggle_input = Path("/kaggle/input/aurelium-openalex-ai-2015-2023/papers.jsonl")
local_raw = Path("../data/raw/papers.jsonl").resolve()
local_alt = Path("data/raw/papers.jsonl").resolve()
local_test = Path("data/test/raw/papers.jsonl").resolve()

if kaggle_input.exists():
    DATA_PATH = str(kaggle_input)
elif local_raw.exists():
    DATA_PATH = str(local_raw)
elif local_alt.exists():
    DATA_PATH = str(local_alt)
elif local_test.exists():
    DATA_PATH = str(local_test)
else:
    DATA_PATH = "papers.jsonl"

print(f"Selected Dataset Path: {DATA_PATH}")

# Ensure ml/ source modules are accessible
for p in [Path(".").resolve(), Path("..").resolve(), Path("../..").resolve()]:
    if (p / "src" / "citation_model.py").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

In [ ]:
# 2. Import Reusable Phase 2 Modules
from src.citation_labels import build_labels, check_coverage, stream_jsonl
from src.embed import embed_corpus
from src.citation_model import run_training_pipeline
from src.interfaces import StubImpact, ImpactModel

print("Successfully imported Phase 2 citation prediction modules.")

In [ ]:
# 3. Diagnostic Coverage Check
# Verifies that older publication cohorts (2015, 2016) have complete observation windows
if os.path.exists(DATA_PATH):
    print(f"Checking cohort observation windows for {DATA_PATH}...")
    coverage_report = check_coverage(stream_jsonl(DATA_PATH))
    print(f"{'Year':<6} | {'Total':<8} | {'% Has Cby':<12} | {'% Non-zero Window':<20} | {'Year Range in Cby'}")
    print("-" * 75)
    for yr, s in coverage_report.items():
        yr_range = f"{s['min_year_seen']} - {s['max_year_seen']}"
        print(f"{yr:<6} | {s['total_papers']:<8} | {s['pct_with_cby']:<11.1f}% | {s['pct_with_window_citations']:<19.1f}% | {yr_range}")
else:
    print(f"Notice: {DATA_PATH} not found. Please attach the Phase 1 dataset.")

In [ ]:
# 4. Load Records and Compute Target Labels
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if os.path.exists(DATA_PATH):
    records = list(stream_jsonl(DATA_PATH))
    print(f"Loaded {len(records):,} records.")
    
    labels_df = build_labels(records)
    print("\nLabels preview:")
    print(labels_df.head())
    
    print("\nSummary statistics for raw citations y:")
    print(labels_df["citations_3yr"].describe())
    
    # Plot citation distribution comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(labels_df["citations_3yr"], bins=40, color="#1f77b4", edgecolor="black", alpha=0.7)
    ax1.set_title("Raw Citation Distribution (Heavily Skewed)")
    ax1.set_xlabel("Citations (3-Year)")
    
    ax2.hist(labels_df["log_citations_3yr"], bins=40, color="#2ca02c", edgecolor="black", alpha=0.7)
    ax2.set_title("Log-Transformed log1p(y) (Bell-Shaped Curve)")
    ax2.set_xlabel("log1p(3-Year Citations)")
    plt.tight_layout()
    plt.show()
else:
    print("Dataset path not found.")

In [ ]:
# 5. Generate or Load Cached SPECTER Embeddings
work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("data")
clean_dir = work_dir / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

cache_npy = str(clean_dir / "embeddings.npy")
cache_ids = str(clean_dir / "embedding_ids.json")

if os.path.exists(DATA_PATH):
    embeddings, ids, has_abs_flags = embed_corpus(
        records=records,
        cache_path=cache_npy,
        ids_path=cache_ids,
        batch_size=64 if torch.cuda.is_available() else 32,
    )
    print(f"Embeddings ready: shape={embeddings.shape}, dtype={embeddings.dtype}")

In [ ]:
# 6. Train Models & Evaluate (LightGBM, PCA, and PyTorch MLP)
models_dir = str(work_dir / "models" / "phase2")
reports_dir = str(work_dir / "reports")

if os.path.exists(DATA_PATH):
    metadata = run_training_pipeline(
        records=records,
        embeddings=embeddings,
        out_models_dir=models_dir,
        out_reports_dir=reports_dir,
        use_pca_comparison=True,
    )
    print("Training pipeline complete!")

In [ ]:
# 7. Display Visualizations and Report
from IPython.display import Image, display, Markdown

scatter_img = Path(reports_dir) / "figures" / "predicted_vs_actual.png"
importance_img = Path(reports_dir) / "figures" / "feature_importance.png"
report_md_file = Path(reports_dir) / "phase2_citation_report.md"

if scatter_img.exists():
    print("--- Predicted vs. Actual Alignment ---")
    display(Image(filename=str(scatter_img)))

if importance_img.exists():
    print("--- Top Predictive Features ---")
    display(Image(filename=str(importance_img)))

if report_md_file.exists():
    with open(report_md_file, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

In [ ]:
# 8. Test ImpactProtocol Interface Prediction
sample_paper = {
    "title": "Scaling Laws for Neural Language Models",
    "abstract": "We investigate empirical scaling laws for language model performance across model size, dataset size, and compute.",
    "publication_date": "2020-01-23",
    "type": "article",
    "field": "Computer Science",
    "subfield": "Artificial Intelligence",
    "topic": "Language Models and Scaling Laws",
    "referenced_works": ["W1", "W2", "W3", "W4", "W5"]
}

impact_model = ImpactModel(model_dir=models_dir)
raw_pred = impact_model.predict(sample_paper)
print(f"Sample Paper: '{sample_paper['title']}'")
print(f"Predicted 3-Year Citation Count: {raw_pred:.2f} citations")

## Kaggle Dataset Export & Downstream Usage

To use the trained Phase 2 artifacts (`lightgbm_model.txt`, `preprocessor.joblib`, `metadata.json`, and `embeddings.npy`) in subsequent phases:

1. Click **"Save Version"** $\to$ **"Save & Run All (Commit)"**.
2. When the run finishes, navigate to the **Output** tab.
3. Click **"New Dataset"** from output and name it `aurelium-phase2-citation-models`.
4. Phase 3 (Embeddings/FAISS) and Phase 5 (Novelty Detection) will attach and reuse `embeddings.npy` directly.